# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [ ]:
from pathlib import Path
from openai import AzureOpenAI, OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import os
from App.config import azure_endpoint, api_version, headers
from sentence_transformers import SentenceTransformer

MODEL = "azure/gpt-4.1-nano"
MODEL_OLLAMA = "ollama/gpt-oss:120b-cloud"

DB_NAME = "week5/preprocessed_db"
collection_name = "docs"
embedding_model = "all-MiniLM-L6-v2" # need to update
KNOWLEDGE_BASE_PATH = Path("week5/knowledge-base")
AVERAGE_CHUNK_SIZE = 500


In [ ]:
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv('cd_api_key_backup')
OLLAMA_API_KEY = os.getenv('OLLAMA_API_KEY')

ollama_url = "https://ollama.com"

if (OPENAI_API_KEY):
    print('OpenAI API key found and looks good so far')
else:
    print('OpenAI API key not found')

if (OLLAMA_API_KEY):
    print('Ollama API key found and looks good so far')
else:
    print('Ollama API key not found')


In [ ]:
# When using openai embeddings

# openai = AzureOpenAI(azure_endpoint=azure_endpoint, api_version=api_version, api_key=OPENAI_API_KEY)

In [ ]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {'source': document['source'], 'type': document['type']}
        return Result(page_content=f'{self.headline} \n\n {self.summary} \n\n {self.original_text}', metadata=metadata)

class Chunks(BaseModel):
    chunks: list[Chunk]


## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [ ]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob('*.md'):
            with open(file, 'r', encoding='utf-8') as fs:
                documents.append({'type': doc_type, 'source': file.as_posix(), 'text': fs.read()})

    print(f'Loaded {len(documents)} documents')
    return documents


In [ ]:
documents = fetch_documents()

### Donezo! On to Step 2 - make the chunks

In [ ]:
def make_prompt(document):
    how_many = (len(document['text']) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document['type']}
The document has been retrieved from: {document['source']}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [ ]:
print(make_prompt(documents[0]))

In [ ]:
def make_messages(document):
    return [
        {'role': 'user', 'content': make_prompt(document)}
    ]

In [ ]:
make_messages(documents[0])

In [ ]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, api_base=azure_endpoint, api_key=OPENAI_API_KEY, api_version=api_version, extra_headers=headers, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    print(reply)
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [ ]:
process_document(documents[0])

In [ ]:
def create_chunks(documents):
    chunks = []

    for doc in tqdm(documents):
        chunks.extend(process_document(doc))

    return chunks

In [ ]:
chunks = create_chunks(documents)

In [ ]:
print(len(chunks))

### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [ ]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)

    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]

    # Convert all texts into embeddings
    emb = SentenceTransformer(embedding_model)
    vectors = emb.encode(texts).tolist()

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, metadatas=metas, embeddings=vectors, documents=texts)
    print(f'Vectorstore created with {collection.count()} documents')


In [ ]:
create_embeddings(chunks)

# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [ ]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing